In [2]:
# Install spaCy
!pip install spacy -q

# Download English language model
!python -m spacy download en_core_web_sm -q

# Install pytesseract for OCR pipeline
!apt-get install tesseract-ocr -qq
!pip install pytesseract pillow -q

# Imports
import re
import json
import spacy
import pytesseract
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from datetime import datetime

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

print('✅ spaCy ready!')
print(f'spaCy version: {spacy.__version__}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ spaCy ready!
spaCy version: 3.8.14


In [3]:
# Part 1: Regular Expressions
print("=" * 50)
print("PART 1: REGULAR EXPRESSIONS")
print("=" * 50)

# Task 1.1: Extract Dates
def extract_dates(text):
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',           # MM/DD/YYYY
        r'\d{1,2}-\d{1,2}-\d{4}',            # DD-MM-YYYY
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',
        r'\d{4}-\d{2}-\d{2}'                  # ISO format
    ]
    dates = []
    for pattern in patterns:
        matches = re.findall(pattern, text)
        dates.extend(matches)
    return dates

# Task 1.2: Extract Amounts
def extract_amounts(text):
    pattern = r'\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
    amounts = re.findall(pattern, text)
    cleaned = []
    for amount in amounts:
        clean = amount.replace('$', '').replace(',', '')
        try:
            cleaned.append(float(clean))
        except:
            pass
    return cleaned

# Task 1.3: Extract Invoice Numbers
def extract_invoice_number(text):
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'#\d{5,}',
        r'ORDER-[A-Z0-9]+',
        r'Invoice (?:Number|#):?\s*([A-Z0-9-]+)'
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1) if match.groups() else match.group(0)
    return None

# Test all functions
test_text = """
Invoice Number: INV-2024-001
Invoice date: 03/15/2024. Due: March 30, 2024
Bill To: John Smith, Acme Corporation
New York, NY 10001

Item 1: Web Development     $1,250.50
Item 2: Design Services     $500.00
Tax:                        $125.05
TOTAL:                      $1,875.55
"""

print("📅 Dates found:")
print(extract_dates(test_text))

print("\n💰 Amounts found:")
print(extract_amounts(test_text))

print("\n🔢 Invoice number:")
print(extract_invoice_number(test_text))
print('\n✅ Regex extraction complete!')

PART 1: REGULAR EXPRESSIONS
📅 Dates found:
['03/15/2024', 'March 30, 2024']

💰 Amounts found:
[202.0, 4.0, 1.0, 3.0, 15.0, 202.0, 4.0, 30.0, 202.0, 4.0, 100.0, 1.0, 1.0, 1250.5, 2.0, 500.0, 125.05, 1875.55]

🔢 Invoice number:
INV-2024-001

✅ Regex extraction complete!


In [4]:
# Cell 3 - Part 2: Named Entity Recognition
import spacy

print("=" * 50)
print("PART 2: NAMED ENTITY RECOGNITION")
print("=" * 50)

# Load spaCy
nlp = spacy.load('en_core_web_sm')

# Sample invoice text
invoice_text = """
Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith (john@acme.com)
Prepared by: Sarah Johnson, Finance Department
Date: March 15, 2024
Due Date: April 15, 2024
Invoice Number: INV-2024-001
Services provided to Microsoft Corporation
Project: Website Redesign for London office
Amount Due: $1,875.55
Payment to: First National Bank
"""

# Process with spaCy
doc = nlp(invoice_text)

# Extract and display entities
print('Found entities:')
print(f"{'Text':<25} {'Label':<15} {'Description'}")
print("-" * 60)
for ent in doc.ents:
    print(f'{ent.text:<25} {ent.label_:<15} {spacy.explain(ent.label_)}')

print('\n✅ NER extraction complete!')

PART 2: NAMED ENTITY RECOGNITION
Found entities:
Text                      Label           Description
------------------------------------------------------------
Acme Corporation          ORG             Companies, agencies, institutions, etc.
123                       CARDINAL        Numerals that do not fall under another type
Main Street               FAC             Buildings, airports, highways, bridges, etc.
New York                  GPE             Countries, cities, states
10001                     DATE            Absolute or relative dates or periods
John Smith                PERSON          People, including fictional
Sarah Johnson             PERSON          People, including fictional
Finance Department        ORG             Companies, agencies, institutions, etc.
March 15, 2024            DATE            Absolute or relative dates or periods
April 15, 2024            DATE            Absolute or relative dates or periods
INV-2024-001              CARDINAL        Numerals

In [5]:
# Cell 4 - Task 2.2: Extract Specific Entity Types
def extract_entities(text):
    """Extract and organize entities by type"""
    doc = nlp(text)

    entities = {
        'persons':       [],
        'organizations': [],
        'locations':     [],
        'dates':         [],
        'money':         []
    }

    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ == 'DATE':
            entities['dates'].append(ent.text)
        elif ent.label_ == 'MONEY':
            entities['money'].append(ent.text)

    return entities

# Test on the same invoice_text from Cell 3
result = extract_entities(invoice_text)

print("Extracted Entities by Type:")
print("=" * 40)
for entity_type, values in result.items():
    print(f'{entity_type:<18}: {values}')

print('\n✅ Entity extraction by type complete!')

Extracted Entities by Type:
persons           : ['John Smith', 'Sarah Johnson']
organizations     : ['Acme Corporation', 'Finance Department', 'Microsoft Corporation\nProject: Website Redesign', 'First National Bank']
locations         : ['New York', 'London']
dates             : ['10001', 'March 15, 2024', 'April 15, 2024']
money             : ['1,875.55']

✅ Entity extraction by type complete!


In [6]:
# Cell 5 - Task 2.3: Visualize Entities with displaCy (Fixed)
from spacy import displacy

# Recreate doc
doc = nlp(invoice_text)

print("Entity Visualization:")
print("=" * 40)

# Call 1: Display inline in Jupyter
displacy.render(doc, style='ent', jupyter=True)

# Call 2: Generate HTML string separately for saving
html = displacy.render(doc, style='ent', page=True, jupyter=False)

# Save to file
with open('entities.html', 'w', encoding='utf-8') as f:
    f.write(html)

print('\n✅ Visualization saved to entities.html')

Entity Visualization:



✅ Visualization saved to entities.html


In [7]:
# Cell 6 - Task 3.1: Complete Invoice Processor
import json

def process_invoice_text(text):
    """
    Complete pipeline: Text → Regex Extraction → NER → JSON
    """
    # Step 1: Extract with regex
    invoice_data = {
        'invoice_number': extract_invoice_number(text),
        'dates':          extract_dates(text),
        'amounts':        extract_amounts(text),
    }

    # Step 2: Extract with NER
    entities = extract_entities(text)
    invoice_data.update(entities)

    # Step 3: Post-process
    if invoice_data['amounts']:
        invoice_data['total_amount'] = max(invoice_data['amounts'])
    if invoice_data['dates']:
        invoice_data['invoice_date'] = invoice_data['dates'][0]

    return invoice_data

# Test on our invoice_text
result = process_invoice_text(invoice_text)

print("Extracted Invoice Data:")
print("=" * 50)
print(json.dumps(result, indent=2))
print('\n✅ Invoice processing complete!')

Extracted Invoice Data:
{
  "invoice_number": "INV-2024-001",
  "dates": [
    "10001",
    "March 15, 2024",
    "April 15, 2024"
  ],
  "amounts": [
    123.0,
    100.0,
    1.0,
    15.0,
    202.0,
    4.0,
    15.0,
    202.0,
    4.0,
    202.0,
    4.0,
    1.0,
    1875.55
  ],
  "persons": [
    "John Smith",
    "Sarah Johnson"
  ],
  "organizations": [
    "Acme Corporation",
    "Finance Department",
    "Microsoft Corporation\nProject: Website Redesign",
    "First National Bank"
  ],
  "locations": [
    "New York",
    "London"
  ],
  "money": [
    "1,875.55"
  ],
  "total_amount": 1875.55,
  "invoice_date": "10001"
}

✅ Invoice processing complete!


In [8]:
# Cell 7 - Task 3.2: Save Results as JSON
import json

# Save to JSON file
output_file = 'extracted_data.json'

with open(output_file, 'w') as f:
    json.dump(result, f, indent=2)

print(f'✅ Results saved to {output_file}')

# Preview what was saved
print('\nPreview of saved JSON:')
print('=' * 50)
with open(output_file, 'r') as f:
    saved_data = json.load(f)

for key, value in saved_data.items():
    print(f'{key:<20}: {value}')

✅ Results saved to extracted_data.json

Preview of saved JSON:
invoice_number      : INV-2024-001
dates               : ['10001', 'March 15, 2024', 'April 15, 2024']
amounts             : [123.0, 100.0, 1.0, 15.0, 202.0, 4.0, 15.0, 202.0, 4.0, 202.0, 4.0, 1.0, 1875.55]
persons             : ['John Smith', 'Sarah Johnson']
organizations       : ['Acme Corporation', 'Finance Department', 'Microsoft Corporation\nProject: Website Redesign', 'First National Bank']
locations           : ['New York', 'London']
money               : ['1,875.55']
total_amount        : 1875.55
invoice_date        : 10001
